In [17]:
import qnexus as qnx
project = qnx.projects.get_or_create(name="CTCs")
qnx.context.set_active_project(project)

In [3]:
# Create a configuration to target the H1-1LE noiseless simulator
my_quantinuum_config = qnx.QuantinuumConfig(
    device_name="H2-1",
)

In [18]:
# from pytket import Circuit

# my_circuit_ref = qnx.circuits.upload(
#     name="My Circuit",
#     circuit=Circuit(2).H(0).CX(0, 1).measure_all(),
#     project=project,
# )

from pytket import Circuit
# from pytket.circuit import Qubit
circuit = Circuit(2)
circuit.H(0).CX(0, 1).measure_all()


[H q[0]; CX q[0], q[1]; Measure q[0] --> c[0]; Measure q[1] --> c[1]; ]

In [19]:
circuit_ref = qnx.circuits.upload(name="MyCircuit", circuit=circuit, project=project)

In [20]:
# For emulator: device_name="H2-1LE"
# For hardware: device_name="H2-1"
config = qnx.QuantinuumConfig(device_name="H2-1")

# Run simulation/compilation
job = qnx.start_execute_job(
    name="H2-1-Run",
    programs=[circuit_ref],
    n_shots=[100],
    backend_config=config,
    project=project
)
# Wait for results
results = job.get_results()

AttributeError: 'ExecuteJobRef' object has no attribute 'get_results'

In [5]:
# Run in an asyncronous manner to receieve a JobRef

compile_job_ref = qnx.start_compile_job(
    programs=[my_circuit_ref],
    name="compile_async",
    optimisation_level=1,
    backend_config=qnx.QuantinuumConfig(device_name="H1-1LE"),
    project=project,
    skip_intermediate_circuits=False,  # Store compiled circuits
)

# Block until the job is complete (or perform other tasks while we wait)
qnx.jobs.wait_for(compile_job_ref)

compiled_circuits = [item.get_output() for item in qnx.jobs.results(compile_job_ref)]

In [6]:
# Retrieve a CompilationResultRef for every Circuit that was compiled
compile_job_result_refs = qnx.jobs.results(compile_job_ref)

compile_job_result_refs.df()

,name,description,created,modified,project,id,job_item_id,job_item_integer_id
0,My Circuit-compilation,,2026-02-27 02:23:17.977805+00:00,2026-02-27 02:23:17.988548+00:00,CTCs,0ab71cc3-d6bb-4f23-b63f-dbc58509d57c,None,1927968


In [7]:
# Retrieve the input CircuitRef for the Circuit
compile_job_result_refs[0].get_input()


# Retrieve the compiled CircuitRef for the Circuit
compile_job_result_refs[0].get_output()


# View the compilation passes that we applied when compiling the Circuit
compile_job_result_refs[0].get_passes().df()

,pass name,input,output,id
0,DecomposeBoxes,My Circuit,My Circuit,abbab95c-c4b3-4ad7-9a94-d57951b87e60
1,CustomPass,My Circuit,My Circuit,b23dc64a-9cbc-4061-bce9-151a9975ca49
2,SynthesiseTK,My Circuit,My Circuit-QuantinuumBackend-2,c4db63f8-1969-4d91-b936-bdc7be81836a
3,NormaliseTK2,My Circuit-QuantinuumBackend-2,My Circuit-QuantinuumBackend-2,5bf4157b-499d-45f2-8e92-86e1438d1173
4,DecomposeTK2,My Circuit-QuantinuumBackend-2,My Circuit-QuantinuumBackend-3,98b61c66-b759-4295-a91d-7b10f4f1b760
5,AutoRebase,My Circuit-QuantinuumBackend-3,My Circuit-QuantinuumBackend-4,e96fdb36-8e40-4303-94fa-7385292f8d95
6,ZZPhaseToRz,My Circuit-QuantinuumBackend-4,My Circuit-QuantinuumBackend-4,971d5ade-0af1-4a4b-aba3-c3e22105f58b
7,RemoveRedundancies,My Circuit-QuantinuumBackend-4,My Circuit-QuantinuumBackend-4,9b3efe95-996f-4474-890a-9aa4348af8d9
8,AutoSquash,My Circuit-QuantinuumBackend-4,My Circuit-QuantinuumBackend-5,d1b6194a-443a-4867-8a4b-6c267c24e22c
9,RemoveRedundancies,My Circuit-QuantinuumBackend-5,My Circuit-QuantinuumBackend-final,4f5c0196-ea1f-4196-afed-9e0b86724fd9


In [16]:
# Run in an asyncronous manner to receieve a JobRef

execute_job_ref = qnx.start_execute_job(
    programs=compiled_circuits,
    name="execute_async",
    n_shots=[1] * len(compiled_circuits),
    backend_config=qnx.QuantinuumConfig(device_name="H2-1"),
    project= project,
)

# Block until the job is complete (or perform other tasks while we wait)
qnx.jobs.wait_for(execute_job_ref)

# Retrieve a ExecutionResultRef for every Circuit that was run
execute_job_result_refs = qnx.jobs.results(execute_job_ref)

execute_job_result_refs.df()

JobError: Job errored with detail: Job cost exceeds allowed cost

In [16]:
# Get the input CircuitRef
execute_job_result_refs[0].get_input()

CircuitRef(id=UUID('67553172-3f34-40a2-864c-de48615ecde5'), annotations=Annotations(name='My Circuit-QuantinuumBackend-final', description=None, properties=OrderedDict(), created=datetime.datetime(2026, 2, 9, 21, 46, 53, 975327, tzinfo=TzInfo(0)), modified=datetime.datetime(2026, 2, 9, 21, 46, 54, 58857, tzinfo=TzInfo(0))), project=ProjectRef(id=UUID('ef977c2a-b90c-4a03-8283-96ddd2c72dd9'), annotations=Annotations(name='CTCs', description='D-CTCs and P-CTCs', properties=OrderedDict(), created=datetime.datetime(2026, 2, 9, 19, 29, 57, 91376, tzinfo=TzInfo(0)), modified=datetime.datetime(2026, 2, 9, 20, 0, 19, 669887, tzinfo=TzInfo(0))), contents_modified=datetime.datetime(2026, 2, 9, 22, 14, 52, 703893, tzinfo=TzInfo(0)), archived=False, type='ProjectRef'), type='CircuitRef')

In [17]:
# Get the results of the execution
result = execute_job_result_refs[0].download_result()

result.get_counts()

Counter({(0, 0): 6, (1, 1): 4})

In [18]:
# Get the pytket BackendInfo to see the state of the device
execute_job_result_refs[0].download_backend_info()

BackendInfo(name='QuantinuumBackend snapshot', device_name='H1-1LE', version='0.54.0', architecture=<tket::FullyConnected, nodes=20>, gate_set={OpType.Barrier, OpType.WASM, OpType.SetBits, OpType.CopyBits, OpType.RangePredicate, OpType.ExplicitPredicate, OpType.ExplicitModifier, OpType.MultiBit, OpType.Rz, OpType.TK2, OpType.Measure, OpType.Reset, OpType.PhasedX, OpType.ZZMax, OpType.ZZPhase, OpType.ClExpr, OpType.RNGSeed, OpType.RNGBound, OpType.RNGIndex, OpType.RNGNum, OpType.JobShotNum}, n_cl_reg=4000, supports_fast_feedforward=True, supports_reset=True, supports_midcircuit_measurement=True, all_node_gate_errors=None, all_edge_gate_errors=None, all_readout_errors=None, averaged_node_gate_errors=None, averaged_edge_gate_errors=None, averaged_readout_errors=None, misc={'batching': False, 'n_shots': 10000, 'options': {}, 'syntax_checker': 'H1-1SC', 'system_type': 'local_emulator', 'wasm': True})

In [24]:
import time
import qnexus as qnx
from pytket.circuit import Circuit

# 1) Login + pick a project
qnx.login()  # browser or prompt auth :contentReference[oaicite:1]{index=1}
project = qnx.projects.get_or_create(name="CTCs")
qnx.context.set_active_project(project)  # :contentReference[oaicite:2]{index=2}

# 2) Build a Bell circuit in pytket
bell = Circuit(2, 2)     # 2 qubits, 2 classical bits
bell.H(0)
bell.CX(0, 1)
bell.Measure(0, 0)
bell.Measure(1, 1)

# 3) Upload circuit to Nexus
suffix = int(time.time())
circuit_ref = qnx.circuits.upload(circuit=bell, name=f"bell-{suffix}")  # :contentReference[oaicite:3]{index=3}

# 4) Compile for real hardware target H2-1
backend = qnx.QuantinuumConfig(device_name="H2-2")  # hardware target string example :contentReference[oaicite:4]{index=4}
compile_job = qnx.start_compile_job(
    programs=[circuit_ref],
    backend_config=backend,
    optimisation_level=3,
    name=f"compile-bell-{suffix}",
)
qnx.jobs.wait_for(compile_job)  # :contentReference[oaicite:5]{index=5}
compiled_circuit_ref = qnx.jobs.results(compile_job)[0].get_output()  # :contentReference[oaicite:6]{index=6}

# 5) Execute on H2-1
shots = 1
exec_job = qnx.start_execute_job(
    programs=[compiled_circuit_ref],
    n_shots=[shots],
    backend_config=backend,
    name=f"exec-bell-{suffix}",
)
qnx.jobs.wait_for(exec_job)  # :contentReference[oaicite:7]{index=7}
result_ref = qnx.jobs.results(exec_job)[0]          # :contentReference[oaicite:8]{index=8}
backend_result = result_ref.download_result()       # :contentReference[oaicite:9]{index=9}

# 6) Read out distribution
dist = backend_result.get_distribution()
print(dist)

Already logged in. Tokens are valid.


JobError: Job errored with detail: Job cost exceeds allowed cost

In [26]:
import qnexus as qnx
qnx.client.quotas.get_all()

/Users/nandan/Desktop/CTCs/nexus/lib/python3.13/site-packages/qnexus/client/__init__.py:131: DeprecationWarning: Your version of qnexus is using a deprecated API endpoint (/api/quotas/v1beta) that will be deleted on Sun, 30 Nov 2025 00:00:00 GMT. After this date your current qnexus version may stop functioning. Please update to a later qnexus version to resolve the issue.
  warnings.warn(


[Quota(name='jupyterhub', description='Total Jupyterhub notebook server running time, in seconds.', usage=24990.0, quota='No quota set for user'),
 Quota(name='simulation', description='Total CPU running time, in seconds.', usage=0.06219078999999983, quota='No quota set for user'),
 Quota(name='database_usage', description='Total megabytes used to store scientific data.', usage=0.3777990000000003, quota='No quota set for user'),
 Quota(name='compilation', description='Total CPU running time, in seconds.', usage=1.5483573230000007, quota='No quota set for user')]

In [27]:
qnx.quotas.Quota()

ValidationError: 4 validation errors for Quota
name
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
description
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
usage
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
quota
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing